# Low-rank evidence adaptation
How many parameters need gradients?

In [ ]:
import os
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(root / 'src'))
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
smoke = os.environ.get('PYTHONIC_SMOKE') == '1'

In [ ]:
from pythonic.model import Decoder, DecoderConfig
from pythonic.lora import inject_lora
model = Decoder(DecoderConfig(width=32, layers=1, context=32))
x = torch.randint(0, 258, (1, 16))
before = model(x).detach()
print('replaced:', inject_lora(model, rank=4))
print('trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('total:', sum(p.numel() for p in model.parameters()))
torch.testing.assert_close(model(x), before)

Zero-initialized output adapters preserve the base model initially. Only the low-rank matrices receive gradients. Reference: [LoRA](https://arxiv.org/abs/2106.09685).